# package

In [ ]:
! pip install fairlearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 45.7 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2


In [ ]:
! pip install tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.3/137.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


# law dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the law_data dataset
law_df = pd.read_csv('/content/law_data.csv')

# Prepare features and target
X = law_df.drop('first_pf', axis=1)
y = law_df['first_pf']

# Sensitive features: race and sex
sensitive_features = {
    'race': law_df['race'],
    'sex': law_df['sex']
}

# Identify numerical and categorical columns
numerical_features = ['LSAT', 'UGPA', 'ZFYA', 'sander_index']
categorical_features = ['region_first']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Define dataset sizes to test
dataset_sizes = [10000, 1000, 500]

# Models to evaluate
models = {
    'LR': Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=42))]),
    'RF': Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('classifier', MLPClassifier(random_state=42, max_iter=300))])
}

# Dictionary to store results
results = {'Dataset Size': [], 'Model': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': []}

# Run experiments for each dataset size
for size in dataset_sizes:
    # Stratified sampling to create subset
    sss = StratifiedShuffleSplit(n_splits=1, train_size=min(size, len(X)), random_state=42)
    for train_index, _ in sss.split(X, y):
        X_subset = X.iloc[train_index]
        y_subset = y.iloc[train_index]
        sens_subset_race = sensitive_features['race'].iloc[train_index]
        sens_subset_sex = sensitive_features['sex'].iloc[train_index]

        # Split subset into train and test
        X_train, X_test, y_train, y_test, sens_train_race, sens_test_race, sens_train_sex, sens_test_sex = train_test_split(
            X_subset, y_subset, sens_subset_race, sens_subset_sex,
            test_size=0.2, random_state=42, stratify=y_subset
        )

        # Evaluate each model
        for name, model in models.items():
            # Fit and predict
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            # Calculate accuracy
            acc = accuracy_score(y_test, y_pred)

            # Fairness metrics for each sensitive feature
            for sens_name, sens_test in [('race', sens_test_race), ('sex', sens_test_sex)]:
                dp_diff = fm.demographic_parity_difference(y_test, y_pred, sensitive_features=sens_test)
                eo_diff = fm.equalized_odds_difference(y_test, y_pred, sensitive_features=sens_test)

                # Store results
                results['Dataset Size'].append(size)
                results['Model'].append(name)
                results['Sensitive'].append(sens_name)
                results['DP Diff'].append(round(dp_diff, 4))
                results['EO Diff'].append(round(eo_diff, 4))
                results['Accuracy'].append(round(acc, 4))

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("Law Dataset Fairness Results for Different Sizes:")
print(results_df)

Law Dataset Fairness Results for Different Sizes:
    Dataset Size Model Sensitive  DP Diff  EO Diff  Accuracy
0          10000    LR      race   0.2168   0.4850    0.8990
1          10000    LR       sex   0.0211   0.0598    0.8990
2          10000    RF      race   0.1857   0.3578    0.8815
3          10000    RF       sex   0.0207   0.0161    0.8815
4          10000   MLP      race   0.2023   0.4109    0.8950
5          10000   MLP       sex   0.0216   0.0343    0.8950
6           1000    LR      race   0.2157   0.5536    0.9000
7           1000    LR       sex   0.0010   0.0855    0.9000
8           1000    RF      race   0.1608   0.3036    0.9000
9           1000    RF       sex   0.0006   0.1197    0.9000
10          1000   MLP      race   0.2255   0.6071    0.8950
11          1000   MLP       sex   0.0085   0.0513    0.8950
12           500    LR      race   0.3524   0.6333    0.9400
13           500    LR       sex   0.0282   0.2917    0.9400
14           500    RF      race   

In [ ]:
"""
Finetuning for Law dataset with fairness evaluation across different dataset sizes.
Subsamples data at specified levels (10000, 1000, 500) instead of using full dataset.
"""
from functools import partial
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.optim import Adam, Optimizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from tabpfn import TabPFNClassifier
from tabpfn.finetune_utils import clone_model_for_evaluation
from tabpfn.utils import meta_dataset_collator
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

def prepare_data(config: dict, dataset_size: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Loads and splits the Law dataset with subsampling."""
    print(f"--- 1. Data Preparation (Size: {dataset_size}) ---")
    law_df = pd.read_csv('/content/law_data.csv')
    X_all = law_df.drop('first_pf', axis=1)
    y_all = law_df['first_pf']
    race_all = law_df['race']
    sex_all = law_df['sex']

    # Subsample data
    sss = StratifiedShuffleSplit(n_splits=1, train_size=min(dataset_size, len(X_all)), random_state=config["random_seed"])
    for train_index, _ in sss.split(X_all, y_all):
        X_subset = X_all.iloc[train_index]
        y_subset = y_all.iloc[train_index]
        race_subset = race_all.iloc[train_index]
        sex_subset = sex_all.iloc[train_index]

    # Preprocessing: Standardize numerical, one-hot categorical
    numerical_features = X_subset.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X_subset.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
        ])
    X_subset_processed = preprocessor.fit_transform(X_subset)

    # Split subset
    splitter = partial(
        train_test_split,
        test_size=config["valid_set_ratio"],
        random_state=config["random_seed"],
    )
    X_train, X_test, y_train, y_test = splitter(X_subset_processed, y_subset, stratify=y_subset)
    _, race_test, _, _ = splitter(race_subset, y_subset, stratify=y_subset)
    _, sex_test, _, _ = splitter(sex_subset, y_subset, stratify=y_subset)

    print(
        f"Loaded subset (size: {dataset_size}) and split: {X_train.shape[0]} train, {X_test.shape[0]} test samples."
    )
    print("---------------------------\n")
    return X_train, X_test, y_train, y_test, race_test, sex_test

def setup_model_and_optimizer(config: dict) -> tuple[TabPFNClassifier, Optimizer, dict]:
    """Initializes the TabPFN classifier, optimizer, and training configs."""
    print("--- 2. Model and Optimizer Setup ---")
    classifier_config = {
        "ignore_pretraining_limits": True,
        "device": config["device"],
        "n_estimators": 2,
        "random_state": config["random_seed"],
        "inference_precision": torch.float32,
    }
    classifier = TabPFNClassifier(
        **classifier_config, fit_mode="batched", differentiable_input=False
    )
    classifier._initialize_model_variables()
    # Optimizer uses finetuning-specific learning rate
    optimizer = Adam(
        classifier.model_.parameters(), lr=config["finetuning"]["learning_rate"]
    )
    print(f"Using device: {config['device']}")
    print(f"Optimizer: Adam, Finetuning LR: {config['finetuning']['learning_rate']}")
    print("----------------------------------\n")
    return classifier, optimizer, classifier_config

def evaluate_model(
    classifier: TabPFNClassifier,
    eval_config: dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    race_test: np.ndarray,
    sex_test: np.ndarray,
) -> tuple[float, float, float, pd.DataFrame]:
    """Evaluates utility and fairness on the test set."""
    eval_classifier = clone_model_for_evaluation(
        classifier, eval_config, TabPFNClassifier
    )
    eval_classifier.fit(X_train, y_train)
    try:
        probabilities = eval_classifier.predict_proba(X_test)
        predictions = (probabilities[:, 1] > 0.5).astype(int)
        roc_auc = roc_auc_score(y_test, probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)
        log_loss_score = log_loss(y_test, probabilities)
        # Fairness metrics
        sensitive = {'race': race_test, 'sex': sex_test}
        results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
        for sens_name, sens_test in sensitive.items():
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
        fairness_df = pd.DataFrame(results)
    except Exception as e:
        print(f"An error occurred during evaluation: {e}")
        roc_auc, accuracy, log_loss_score = np.nan, np.nan, np.nan
        fairness_df = pd.DataFrame()
    return roc_auc, accuracy, log_loss_score, fairness_df

def main() -> None:
    """Main function to configure and run the finetuning workflow with fairness eval for different dataset sizes."""
    # --- Master Configuration ---
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,  # Reduced for full data to save memory
    }
    config["finetuning"] = {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 5000,  # Adjusted for subsampled data
    }

    # Define dataset sizes to test
    dataset_sizes = [10000, 1000, 500]

    # Loop over dataset sizes
    for dataset_size in dataset_sizes:
        print(f"\n=== Processing Dataset Size: {dataset_size} ===\n")
        # --- Setup Data, Model, and Dataloader ---
        X_train, X_test, y_train, y_test, race_test, sex_test = prepare_data(config, dataset_size)
        classifier, optimizer, classifier_config = setup_model_and_optimizer(config)
        splitter = partial(train_test_split, test_size=config["valid_set_ratio"])
        training_datasets = classifier.get_preprocessed_datasets(
            X_train, y_train, splitter, config["finetuning"]["batch_size"]
        )
        finetuning_dataloader = DataLoader(
            training_datasets,
            batch_size=config["finetuning"]["meta_batch_size"],
            collate_fn=meta_dataset_collator,
        )
        loss_function = torch.nn.CrossEntropyLoss()
        eval_config = {
            **classifier_config,
            "inference_config": {
                "SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]
            },
        }

        # --- Finetuning and Evaluation Loop ---
        print("--- 3. Starting Finetuning & Evaluation ---")
        for epoch in range(config["finetuning"]["epochs"] + 1):
            if epoch > 0:
                # Finetuning Step
                progress_bar = tqdm(finetuning_dataloader, desc=f"Finetuning Epoch {epoch} (Size: {dataset_size})")
                for (
                    X_train_batch,
                    X_test_batch,
                    y_train_batch,
                    y_test_batch,
                    cat_ixs,
                    confs,
                ) in progress_bar:
                    if len(np.unique(y_train_batch)) != len(np.unique(y_test_batch)):
                        continue
                    optimizer.zero_grad()
                    classifier.fit_from_preprocessed(
                        X_train_batch, y_train_batch, cat_ixs, confs
                    )
                    predictions = classifier.forward(X_test_batch, return_logits=True)
                    loss = loss_function(predictions, y_test_batch.to(config["device"]))
                    loss.backward()
                    optimizer.step()
                    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

            # Evaluation Step
            epoch_roc, epoch_acc, epoch_log_loss, fairness_df = evaluate_model(
                classifier, eval_config, X_train, y_train, X_test, y_test, race_test, sex_test
            )
            status = "Initial" if epoch == 0 else f"Epoch {epoch}"
            print(
                f"📊 {status} Utility (Size: {dataset_size}) | Test ROC: {epoch_roc:.4f}, Accuracy: {epoch_acc:.4f}, Log Loss: {epoch_log_loss:.4f}\n"
            )
            print(f"{status} Fairness Results (Size: {dataset_size}):\n{fairness_df}\n")

        print(f"--- ✅ Finetuning Finished for Size {dataset_size} ---")

if __name__ == "__main__":
    main()


=== Processing Dataset Size: 10000 ===

--- 1. Data Preparation (Size: 10000) ---
Loaded subset (size: 10000) and split: 7000 train, 3000 test samples.
---------------------------

--- 2. Model and Optimizer Setup ---
Using device: cuda
Optimizer: Adam, Finetuning LR: 1e-05
----------------------------------

--- 3. Starting Finetuning & Evaluation ---
📊 Initial Utility (Size: 10000) | Test ROC: 0.8294, Accuracy: 0.8980, Log Loss: 0.2765

Initial Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.1731   0.4059
1       sex   0.0125   0.0093



Finetuning Epoch 1 (Size: 10000): 100%|██████████| 2/2 [00:02<00:00,  1.27s/it, loss=0.2555]


📊 Epoch 1 Utility (Size: 10000) | Test ROC: 0.8296, Accuracy: 0.8980, Log Loss: 0.2759

Epoch 1 Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.2044   0.4534
1       sex   0.0147   0.0099



Finetuning Epoch 2 (Size: 10000): 100%|██████████| 2/2 [00:02<00:00,  1.26s/it, loss=0.2453]


📊 Epoch 2 Utility (Size: 10000) | Test ROC: 0.8294, Accuracy: 0.8967, Log Loss: 0.2753

Epoch 2 Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.2078   0.4534
1       sex   0.0138   0.0089



...

📊 Epoch 8 Utility (Size: 500) | Test ROC: 0.8987, Accuracy: 0.9267, Log Loss: 0.2229

Epoch 8 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.3601   0.6389
1       sex   0.0339   0.1364



Finetuning Epoch 9 (Size: 500): 100%|██████████| 1/1 [00:00<00:00,  3.23it/s, loss=0.2810]


📊 Epoch 9 Utility (Size: 500) | Test ROC: 0.8983, Accuracy: 0.9200, Log Loss: 0.2233

Epoch 9 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.4226   0.6389
1       sex   0.0214   0.1364



Finetuning Epoch 10 (Size: 500): 100%|██████████| 1/1 [00:00<00:00,  3.20it/s, loss=0.2720]


📊 Epoch 10 Utility (Size: 500) | Test ROC: 0.8978, Accuracy: 0.9200, Log Loss: 0.2237

Epoch 10 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.4226   0.6389
1       sex   0.0214   0.1364

--- ✅ Finetuning Finished for Size 500 ---


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the law_data dataset
law_df = pd.read_csv('/content/law_data.csv')

# Prepare features and target
X = law_df.drop('first_pf', axis=1)
y = law_df['first_pf']

# Sensitive features: race and sex
sensitive_features = {
    'race': law_df['race'],
    'sex': law_df['sex']
}

# Identify numerical and categorical columns
numerical_features = ['LSAT', 'UGPA', 'ZFYA', 'sander_index', 'race', 'sex']
categorical_features = ['region_first']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

# Define dataset sizes to test
dataset_sizes = [10000, 1000, 500]

# Dictionary to store results
results = {'Dataset Size': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': [], 'ROC AUC': []}

# Run experiments for each dataset size
for size in dataset_sizes:
    # Stratified sampling to create subset
    sss = StratifiedShuffleSplit(n_splits=1, train_size=min(size, len(X)), random_state=42)
    for train_index, _ in sss.split(X, y):
        X_subset = X.iloc[train_index]
        y_subset = y.iloc[train_index]
        sens_subset_race = sensitive_features['race'].iloc[train_index]
        sens_subset_sex = sensitive_features['sex'].iloc[train_index]

        # Apply preprocessing
        X_subset_processed = preprocessor.fit_transform(X_subset)

        # Split subset into train and test
        X_train, X_test, y_train, y_test, sens_train_race, sens_test_race, sens_train_sex, sens_test_sex = train_test_split(
            X_subset_processed, y_subset, sens_subset_race, sens_subset_sex,
            test_size=0.5, random_state=42, stratify=y_subset
        )

        # Initialize and fit the classifier
        clf = TabPFNClassifier(ignore_pretraining_limits=True)
        clf.fit(X_train, y_train)

        # Predict probabilities and labels
        prediction_probabilities = clf.predict_proba(X_test)
        predictions = clf.predict(X_test)

        # Calculate performance metrics
        roc_auc = roc_auc_score(y_test, prediction_probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)

        # Fairness metrics
        for sens_name, sens_test in [('race', sens_test_race), ('sex', sens_test_sex)]:
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)

            # Store results
            results['Dataset Size'].append(size)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
            results['Accuracy'].append(round(accuracy, 4))
            results['ROC AUC'].append(round(roc_auc, 4))

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("Law Dataset TabPFN Fairness Results for Different Sizes:")
print(results_df)

Law Dataset TabPFN Fairness Results for Different Sizes:
   Dataset Size Sensitive  DP Diff  EO Diff  Accuracy  ROC AUC
0         10000      race   0.1814   0.3379    0.8966   0.8362
1         10000       sex   0.0133   0.0127    0.8966   0.8362
2          1000      race   0.2560   0.5625    0.9040   0.8692
3          1000       sex   0.0191   0.2271    0.9040   0.8692
4           500      race   0.0542   0.0872    0.9000   0.8756
5           500       sex   0.0088   0.1696    0.9000   0.8756


# adult dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the adult dataset
adult_df = pd.read_csv('/content/adult.csv')

# Prepare features and target
X = adult_df.drop('income>50K', axis=1)
y = adult_df['income>50K']

# Sensitive features: race and sex
sensitive_features = {
    'race': adult_df['race'],
    'sex': adult_df['sex']
}

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

# Define dataset sizes to test
dataset_sizes = [10000, 1000, 500]

# Dictionary to store results
results = {'Dataset Size': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': [], 'ROC AUC': []}

# Run experiments for each dataset size
for size in dataset_sizes:
    # Stratified sampling to create subset
    sss = StratifiedShuffleSplit(n_splits=1, train_size=size, random_state=42)
    for train_index, _ in sss.split(X, y):
        X_subset = X.iloc[train_index]
        y_subset = y.iloc[train_index]
        sens_subset_race = sensitive_features['race'].iloc[train_index]
        sens_subset_sex = sensitive_features['sex'].iloc[train_index]

        # Apply preprocessing
        X_subset_processed = preprocessor.fit_transform(X_subset)

        # Split subset into train and test
        X_train, X_test, y_train, y_test, sens_train_race, sens_test_race, sens_train_sex, sens_test_sex = train_test_split(
            X_subset_processed, y_subset, sens_subset_race, sens_subset_sex,
            test_size=0.5, random_state=42, stratify=y_subset
        )

        # Initialize and fit the classifier
        clf = TabPFNClassifier(ignore_pretraining_limits=True)
        clf.fit(X_train, y_train)

        # Predict probabilities and labels
        prediction_probabilities = clf.predict_proba(X_test)
        predictions = clf.predict(X_test)

        # Calculate performance metrics
        roc_auc = roc_auc_score(y_test, prediction_probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)

        # Fairness metrics
        for sens_name, sens_test in [('race', sens_test_race), ('sex', sens_test_sex)]:
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)

            # Store results
            results['Dataset Size'].append(size)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
            results['Accuracy'].append(round(accuracy, 4))
            results['ROC AUC'].append(round(roc_auc, 4))

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("Adult Dataset TabPFN Fairness Results for Different Sizes:")
print(results_df)

tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

Adult Dataset TabPFN Fairness Results for Different Sizes:
   Dataset Size Sensitive  DP Diff  EO Diff  Accuracy  ROC AUC
0         10000      race   0.2417   0.7027    0.8634   0.9147
1         10000       sex   0.1690   0.0611    0.8634   0.9147
2          1000      race   0.2000   0.5534    0.8560   0.8957
3          1000       sex   0.1535   0.0529    0.8560   0.8957
4           500      race   0.3551   0.5577    0.8320   0.8845
5           500       sex   0.2110   0.5273    0.8320   0.8845


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the adult dataset
adult_df = pd.read_csv('/content/adult.csv')

# Prepare features and target
X = adult_df.drop('income>50K', axis=1)
y = adult_df['income>50K']

# Sensitive features: race and sex
sensitive_features = {
    'race': adult_df['race'],
    'sex': adult_df['sex']
}

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline for non-TabPFN models
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Define dataset sizes to test
dataset_sizes = [10000, 1000, 500]

# Models to evaluate
models = {
    'LR': Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=42))]),
    'RF': Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('classifier', MLPClassifier(random_state=42, max_iter=300))]),
}

# Dictionary to store results
results = {'Dataset Size': [], 'Model': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': []}

# Run experiments for each dataset size
for size in dataset_sizes:
    # Stratified sampling to create subset
    sss = StratifiedShuffleSplit(n_splits=1, train_size=size, random_state=42)
    for train_index, _ in sss.split(X, y):
        X_subset = X.iloc[train_index]
        y_subset = y.iloc[train_index]
        sens_subset_race = sensitive_features['race'].iloc[train_index]
        sens_subset_sex = sensitive_features['sex'].iloc[train_index]

        # Split subset into train and test
        X_train, X_test, y_train, y_test, sens_train_race, sens_test_race, sens_train_sex, sens_test_sex = train_test_split(
            X_subset, y_subset, sens_subset_race, sens_subset_sex,
            test_size=0.2, random_state=42, stratify=y_subset
        )

        # Evaluate each model
        for name, model in models.items():
            # Fit and predict
            if name == 'TabPFN':
                # TabPFN handles preprocessing internally
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

            # Calculate accuracy
            acc = accuracy_score(y_test, y_pred)

            # Fairness metrics for each sensitive feature
            for sens_name, sens_test in [('race', sens_test_race), ('sex', sens_test_sex)]:
                dp_diff = fm.demographic_parity_difference(y_test, y_pred, sensitive_features=sens_test)
                eo_diff = fm.equalized_odds_difference(y_test, y_pred, sensitive_features=sens_test)

                # Store results
                results['Dataset Size'].append(size)
                results['Model'].append(name)
                results['Sensitive'].append(sens_name)
                results['DP Diff'].append(round(dp_diff, 4))
                results['EO Diff'].append(round(eo_diff, 4))
                results['Accuracy'].append(round(acc, 4))

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("Adult Dataset Fairness Results for Different Sizes:")
print(results_df)

Adult Dataset Fairness Results for Different Sizes:
    Dataset Size Model Sensitive  DP Diff  EO Diff  Accuracy
0          10000    LR      race   0.2241   0.5804    0.8430
1          10000    LR       sex   0.2142   0.2350    0.8430
2          10000    RF      race   0.1914   0.6410    0.8565
3          10000    RF       sex   0.2002   0.1118    0.8565
4          10000   MLP      race   0.2086   0.6247    0.8430
5          10000   MLP       sex   0.2053   0.1138    0.8430
6           1000    LR      race   0.1503   0.5000    0.8450
7           1000    LR       sex   0.1917   0.2906    0.8450
8           1000    RF      race   0.1734   0.5116    0.8300
9           1000    RF       sex   0.1938   0.3162    0.8300
10          1000   MLP      race   0.1734   0.5581    0.8550
11          1000   MLP       sex   0.2057   0.3675    0.8550
12           500    LR      race   0.4000   0.6667    0.8300
13           500    LR       sex   0.1757   0.4545    0.8300
14           500    RF      race 

In [ ]:
"""
Finetuning for Adult dataset with fairness evaluation across different dataset sizes.
Subsamples data at specified levels (10000, 1000, 500) instead of using full dataset.
"""
from functools import partial
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.optim import Adam, Optimizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from tabpfn import TabPFNClassifier
from tabpfn.finetune_utils import clone_model_for_evaluation
from tabpfn.utils import meta_dataset_collator
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

def prepare_data(config: dict, dataset_size: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Loads and splits the Adult dataset with subsampling."""
    print(f"--- 1. Data Preparation (Size: {dataset_size}) ---")
    adult_df = pd.read_csv('/content/adult.csv')
    X_all = adult_df.drop('income>50K', axis=1)
    y_all = adult_df['income>50K']
    race_all = adult_df['race']
    sex_all = adult_df['sex']

    # Subsample data
    sss = StratifiedShuffleSplit(n_splits=1, train_size=min(dataset_size, len(X_all)), random_state=config["random_seed"])
    for train_index, _ in sss.split(X_all, y_all):
        X_subset = X_all.iloc[train_index]
        y_subset = y_all.iloc[train_index]
        race_subset = race_all.iloc[train_index]
        sex_subset = sex_all.iloc[train_index]

    # Preprocessing: Standardize numerical, one-hot categorical
    numerical_features = X_subset.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X_subset.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
        ])
    X_subset_processed = preprocessor.fit_transform(X_subset)

    # Split subset
    splitter = partial(
        train_test_split,
        test_size=config["valid_set_ratio"],
        random_state=config["random_seed"],
    )
    X_train, X_test, y_train, y_test = splitter(X_subset_processed, y_subset, stratify=y_subset)
    _, race_test, _, _ = splitter(race_subset, y_subset, stratify=y_subset)
    _, sex_test, _, _ = splitter(sex_subset, y_subset, stratify=y_subset)

    print(
        f"Loaded subset (size: {dataset_size}) and split: {X_train.shape[0]} train, {X_test.shape[0]} test samples."
    )
    print("---------------------------\n")
    return X_train, X_test, y_train, y_test, race_test, sex_test

def setup_model_and_optimizer(config: dict) -> tuple[TabPFNClassifier, Optimizer, dict]:
    """Initializes the TabPFN classifier, optimizer, and training configs."""
    print("--- 2. Model and Optimizer Setup ---")
    classifier_config = {
        "ignore_pretraining_limits": True,
        "device": config["device"],
        "n_estimators": 2,
        "random_state": config["random_seed"],
        "inference_precision": torch.float32,
    }
    classifier = TabPFNClassifier(
        **classifier_config, fit_mode="batched", differentiable_input=False
    )
    classifier._initialize_model_variables()
    # Optimizer uses finetuning-specific learning rate
    optimizer = Adam(
        classifier.model_.parameters(), lr=config["finetuning"]["learning_rate"]
    )
    print(f"Using device: {config['device']}")
    print(f"Optimizer: Adam, Finetuning LR: {config['finetuning']['learning_rate']}")
    print("----------------------------------\n")
    return classifier, optimizer, classifier_config

def evaluate_model(
    classifier: TabPFNClassifier,
    eval_config: dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    race_test: np.ndarray,
    sex_test: np.ndarray,
) -> tuple[float, float, float, pd.DataFrame]:
    """Evaluates utility and fairness on the test set."""
    eval_classifier = clone_model_for_evaluation(
        classifier, eval_config, TabPFNClassifier
    )
    eval_classifier.fit(X_train, y_train)
    try:
        probabilities = eval_classifier.predict_proba(X_test)
        predictions = (probabilities[:, 1] > 0.5).astype(int)
        roc_auc = roc_auc_score(y_test, probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)
        log_loss_score = log_loss(y_test, probabilities)
        # Fairness metrics
        sensitive = {'race': race_test, 'sex': sex_test}
        results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
        for sens_name, sens_test in sensitive.items():
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
        fairness_df = pd.DataFrame(results)
    except Exception as e:
        print(f"An error occurred during evaluation: {e}")
        roc_auc, accuracy, log_loss_score = np.nan, np.nan, np.nan
        fairness_df = pd.DataFrame()
    return roc_auc, accuracy, log_loss_score, fairness_df

def main() -> None:
    """Main function to configure and run the finetuning workflow with fairness eval for different dataset sizes."""
    # --- Master Configuration ---
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,  # Reduced for full data to save memory
    }
    config["finetuning"] = {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 5000,  # Adjusted for subsampled data
    }

    # Define dataset sizes to test
    dataset_sizes = [10000, 1000, 500]

    # Loop over dataset sizes
    for dataset_size in dataset_sizes:
        print(f"\n=== Processing Dataset Size: {dataset_size} ===\n")
        # --- Setup Data, Model, and Dataloader ---
        X_train, X_test, y_train, y_test, race_test, sex_test = prepare_data(config, dataset_size)
        classifier, optimizer, classifier_config = setup_model_and_optimizer(config)
        splitter = partial(train_test_split, test_size=config["valid_set_ratio"])
        training_datasets = classifier.get_preprocessed_datasets(
            X_train, y_train, splitter, config["finetuning"]["batch_size"]
        )
        finetuning_dataloader = DataLoader(
            training_datasets,
            batch_size=config["finetuning"]["meta_batch_size"],
            collate_fn=meta_dataset_collator,
        )
        loss_function = torch.nn.CrossEntropyLoss()
        eval_config = {
            **classifier_config,
            "inference_config": {
                "SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]
            },
        }

        # --- Finetuning and Evaluation Loop ---
        print("--- 3. Starting Finetuning & Evaluation ---")
        for epoch in range(config["finetuning"]["epochs"] + 1):
            if epoch > 0:
                # Finetuning Step
                progress_bar = tqdm(finetuning_dataloader, desc=f"Finetuning Epoch {epoch} (Size: {dataset_size})")
                for (
                    X_train_batch,
                    X_test_batch,
                    y_train_batch,
                    y_test_batch,
                    cat_ixs,
                    confs,
                ) in progress_bar:
                    if len(np.unique(y_train_batch)) != len(np.unique(y_test_batch)):
                        continue
                    optimizer.zero_grad()
                    classifier.fit_from_preprocessed(
                        X_train_batch, y_train_batch, cat_ixs, confs
                    )
                    predictions = classifier.forward(X_test_batch, return_logits=True)
                    loss = loss_function(predictions, y_test_batch.to(config["device"]))
                    loss.backward()
                    optimizer.step()
                    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

            # Evaluation Step
            epoch_roc, epoch_acc, epoch_log_loss, fairness_df = evaluate_model(
                classifier, eval_config, X_train, y_train, X_test, y_test, race_test, sex_test
            )
            status = "Initial" if epoch == 0 else f"Epoch {epoch}"
            print(
                f"📊 {status} Utility (Size: {dataset_size}) | Test ROC: {epoch_roc:.4f}, Accuracy: {epoch_acc:.4f}, Log Loss: {epoch_log_loss:.4f}\n"
            )
            print(f"{status} Fairness Results (Size: {dataset_size}):\n{fairness_df}\n")

        print(f"--- ✅ Finetuning Finished for Size {dataset_size} ---")

if __name__ == "__main__":
    main()


=== Processing Dataset Size: 10000 ===

--- 1. Data Preparation (Size: 10000) ---
Loaded subset (size: 10000) and split: 7000 train, 3000 test samples.
---------------------------

--- 2. Model and Optimizer Setup ---
Using device: cuda
Optimizer: Adam, Finetuning LR: 1e-05
----------------------------------

--- 3. Starting Finetuning & Evaluation ---
📊 Initial Utility (Size: 10000) | Test ROC: 0.9118, Accuracy: 0.8537, Log Loss: 0.3103

Initial Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.1560   0.6364
1       sex   0.1869   0.0727



Finetuning Epoch 1 (Size: 10000): 100%|██████████| 2/2 [00:02<00:00,  1.48s/it, loss=0.3252]


📊 Epoch 1 Utility (Size: 10000) | Test ROC: 0.9117, Accuracy: 0.8543, Log Loss: 0.3095

Epoch 1 Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.1560   0.6364
1       sex   0.1845   0.0709



Finetuning Epoch 2 (Size: 10000): 100%|██████████| 2/2 [00:02<00:00,  1.31s/it, loss=0.2912]


📊 Epoch 2 Utility (Size: 10000) | Test ROC: 0.9119, Accuracy: 0.8550, Log Loss: 0.3090

Epoch 2 Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.1560   0.6364
1       sex   0.1825   0.0688



...

📊 Epoch 8 Utility (Size: 500) | Test ROC: 0.8816, Accuracy: 0.8400, Log Loss: 0.3546

Epoch 8 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.2981   0.7500
1       sex   0.2104   0.5938



Finetuning Epoch 9 (Size: 500): 100%|██████████| 1/1 [00:00<00:00,  3.15it/s, loss=0.3937]


📊 Epoch 9 Utility (Size: 500) | Test ROC: 0.8816, Accuracy: 0.8400, Log Loss: 0.3543

Epoch 9 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.2981   0.7500
1       sex   0.2104   0.5938



Finetuning Epoch 10 (Size: 500): 100%|██████████| 1/1 [00:00<00:00,  3.25it/s, loss=0.3437]


📊 Epoch 10 Utility (Size: 500) | Test ROC: 0.8813, Accuracy: 0.8400, Log Loss: 0.3542

Epoch 10 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.2981   0.7500
1       sex   0.2104   0.5938

--- ✅ Finetuning Finished for Size 500 ---
